# Multi-Head Self-Attention from Scratch
### Pure NumPy — No PyTorch, No TensorFlow

> **Focus:** Build the attention block in complete isolation, get every masking variant
> airtight, and understand the exact shape transformations at every step — before wiring
> it into a full Transformer.

---

## Roadmap

| # | Section | Concept |
|---|---------|---------|
| 1 | Dot-Product Attention (scalar demo) | The core scoring mechanism |
| 2 | Scaled Dot-Product Attention (single head) | Why we scale by √dₖ |
| 3 | Masking — Padding mask | Ignore `<PAD>` tokens |
| 4 | Masking — Causal (look-ahead) mask | Decoder auto-regression |
| 5 | Masking — Combined mask | Padding + causal together |
| 6 | Multi-Head Self-Attention | Split → attend → concat → project |
| 7 | Shape trace — every tensor annotated | Debug-friendly walkthrough |
| 8 | Numerical checks & symmetry tests | Verify correctness |
| 9 | Visualising attention weights | Heatmaps for each head |
|10 | Cross-Attention (encoder-decoder) | Bonus: Q from decoder, K/V from encoder |

## 0 · Imports & Reproducibility

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

np.random.seed(42)

# Pretty-print helper
def shape(name, arr):
    print(f"  {name:<30} {str(arr.shape)}")

## 1 · Intuition — What Is Attention?

Before any code, the mental model:

**The library metaphor**

| Attention component | Library analogy |
|--------------------|-----------------|
| **Query (Q)** | Your search query: *"books about neural networks"* |
| **Key (K)** | Index card title on each book's spine |
| **Value (V)** | The actual content inside the book |

You match your Query against every Key to get a relevance score.  
You then take a **weighted average of Values**, where the weights come from those scores.

```
Attention(Q, K, V)  =  softmax( Q Kᵀ / √dₖ )  ·  V
```

**Self-attention** means Q, K, and V all come from the *same* sequence —
every token asks "which other tokens are most relevant to me?"

**Multi-head attention** runs this process in parallel `h` times with different
learned projections, letting the model attend to different relationship types
simultaneously (syntax in head 1, coreference in head 2, etc.).

## 2 · Building Blocks

In [ ]:
# ── Numerically stable softmax ───────────────────────────────────────────────
def softmax(x, axis=-1):
    """
    Subtract row-max before exp to avoid overflow.
    softmax(x) = exp(x - max(x)) / Σ exp(x - max(x))

    This is mathematically identical to the standard definition but never
    produces inf/nan for large x values.
    """
    x_max = np.max(x, axis=axis, keepdims=True)   # (batch, heads, seq, 1)
    e     = np.exp(x - x_max)
    return e / np.sum(e, axis=axis, keepdims=True)


# Quick sanity check
logits = np.array([[1.0, 2.0, 3.0],
                   [1000.0, 1001.0, 1002.0]])   # would overflow naive exp
probs  = softmax(logits, axis=-1)
print("Softmax outputs (should sum to 1.0 per row):")
print(probs)
print("Row sums:", probs.sum(axis=-1))  # must be [1.0, 1.0]

## 3 · Scaled Dot-Product Attention (Single Head)

```
Attention(Q, K, V) = softmax( Q Kᵀ / √dₖ ) · V
```

### Why √dₖ scaling?

For a query and key both drawn from Normal(0, 1) with dimension dₖ,
their dot product has:
- Mean = 0
- **Variance = dₖ**

Without scaling, large dₖ → large dot products → softmax in saturation region →
**gradients vanish** (the winning logit dominates, all others → 0).

Dividing by √dₖ restores unit variance regardless of dₖ.

### Shapes

```
Q : (batch, seq_q, d_k)
K : (batch, seq_k, d_k)
V : (batch, seq_k, d_v)

scores   = Q @ Kᵀ         → (batch, seq_q, seq_k)   raw dot products
scores  /= √dₖ            → scaled
scores  += mask            → −∞ on forbidden positions
weights  = softmax(scores) → (batch, seq_q, seq_k)   attention weights
output   = weights @ V     → (batch, seq_q, d_v)     context vectors
```

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Parameters
    ----------
    Q    : (batch, seq_q, d_k)
    K    : (batch, seq_k, d_k)
    V    : (batch, seq_k, d_v)
    mask : (batch, 1, seq_q, seq_k) or (batch, 1, 1, seq_k)
           Boolean array — True means MASK THIS POSITION (set to -inf).

    Returns
    -------
    output  : (batch, seq_q, d_v)   weighted sum of values
    weights : (batch, seq_q, seq_k) attention weights (after softmax)
    """
    d_k = Q.shape[-1]

    # ── Step 1: raw attention scores ──────────────────────────────────────
    # Q @ Kᵀ  →  (batch, seq_q, seq_k)
    scores = Q @ K.transpose(0, 2, 1)          # (batch, seq_q, seq_k)

    # ── Step 2: scale ─────────────────────────────────────────────────────
    scores = scores / np.sqrt(d_k)

    # ── Step 3: apply mask (add -1e9 to forbidden positions) ──────────────
    if mask is not None:
        # mask==True  → position is forbidden → set score to -inf
        # After softmax, exp(-inf) = 0, so masked positions get zero weight
        scores = np.where(mask, -1e9, scores)

    # ── Step 4: softmax over key dimension ────────────────────────────────
    weights = softmax(scores, axis=-1)         # (batch, seq_q, seq_k)

    # ── Step 5: weighted sum of values ────────────────────────────────────
    output  = weights @ V                      # (batch, seq_q, d_v)

    return output, weights


# ── Demo ──────────────────────────────────────────────────────────────────────
batch, seq, d_k, d_v = 1, 4, 8, 8
Q = np.random.randn(batch, seq, d_k)
K = np.random.randn(batch, seq, d_k)
V = np.random.randn(batch, seq, d_v)

output, weights = scaled_dot_product_attention(Q, K, V)
print("Single-head attention (no mask):")
shape("Q", Q); shape("K", K); shape("V", V)
shape("attention weights", weights); shape("output", output)
print()
print("Weights row sums (must all be 1.0):", weights[0].sum(axis=-1).round(6))

---
## 4 · Masking — The Most Critical Part

Masking is what makes attention usable in practice.  
There are **three distinct mask types**, each solving a different problem.

| Mask type | Problem it solves | Used in |
|-----------|-------------------|---------|
| **Padding mask** | Sequences padded to equal length — ignore `<PAD>` | Encoder + Decoder |
| **Causal (look-ahead) mask** | Decoder must not see future tokens | Decoder self-attn |
| **Combined mask** | Both problems simultaneously | Decoder self-attn |

### Convention used throughout this notebook

```
mask value = True  →  FORBIDDEN  →  score set to −1e9  →  softmax weight ≈ 0
mask value = False →  ALLOWED    →  score unchanged
```

### 4a · Padding Mask

**Problem**: batches require equal-length sequences. Shorter sequences are padded:
```
Sentence 1: ["the", "king", "rules",  "<PAD>", "<PAD>"]
Sentence 2: ["paris", "is",  "great",  "city",  "<PAD>"]
```

Token index for `<PAD>` is conventionally 0.

**What goes wrong without it**: `<PAD>` tokens get non-zero attention weights,
which corrupts the context vector — the model learns from noise.

**Shape**: `(batch, 1, 1, seq_k)`  
The `1, 1` broadcasting dims expand to cover all heads and all query positions.

In [ ]:
def make_padding_mask(token_ids, pad_id=0):
    """
    Parameters
    ----------
    token_ids : (batch, seq)  integer token ids
    pad_id    : int           which token id is <PAD>

    Returns
    -------
    mask : (batch, 1, 1, seq)  bool — True where token is <PAD>

    The shape (batch, 1, 1, seq_k) broadcasts over:
      - heads   dim (will be h   after splitting)
      - seq_q   dim (every query avoids every pad key)
    """
    # True where the token IS a pad token
    mask = (token_ids == pad_id)            # (batch, seq)
    return mask[:, np.newaxis, np.newaxis, :]  # (batch, 1, 1, seq)


# ── Demo ──────────────────────────────────────────────────────────────────────
token_ids = np.array([
    [5, 3, 8, 0, 0],   # last 2 are <PAD>
    [2, 7, 1, 4, 0],   # last 1 is  <PAD>
])

pad_mask = make_padding_mask(token_ids, pad_id=0)
print("token_ids shape :", token_ids.shape)
print("padding mask shape:", pad_mask.shape)
print()
print("Padding mask (batch 0) — True = masked out:")
print(pad_mask[0, 0, 0])   # [F, F, F, T, T]
print()
print("Padding mask (batch 1) — True = masked out:")
print(pad_mask[1, 0, 0])   # [F, F, F, F, T]

# ── Visualise ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(9, 2))
for i, ax in enumerate(axes):
    # Expand to (seq_q, seq_k) for display
    display = np.broadcast_to(pad_mask[i, 0], (5, 5)).astype(float)
    im = ax.imshow(display, cmap='RdYlGn_r', vmin=0, vmax=1, aspect='auto')
    ax.set_title(f'Padding mask — batch {i}', fontsize=10)
    ax.set_xlabel("Key position"); ax.set_ylabel("Query position")
    ax.set_xticks(range(5)); ax.set_yticks(range(5))
    tokens = ['tok','tok','tok','<PAD>','<PAD>'] if i==0 else ['tok','tok','tok','tok','<PAD>']
    ax.set_xticklabels(tokens, fontsize=7, rotation=30)
plt.colorbar(im, ax=axes, label='1=masked', shrink=0.8)
plt.suptitle("Padding Mask  (green=attend, red=block)", y=1.02)
plt.tight_layout()
plt.show()

### 4b · Causal (Look-Ahead) Mask

**Problem**: the Transformer decoder generates tokens **one at a time**, left to right.  
During training we feed the full target sequence in parallel (teacher forcing),
but each position must only attend to **previous positions** — not future ones.

If position 3 could see position 5 during training, it would just copy the answer.
At inference time position 5 doesn't exist yet — so the model would fail.

**Solution**: mask the **upper triangle** of the attention score matrix.

```
Position:     0    1    2    3    4
         0  [  ✓    ✗    ✗    ✗    ✗ ]   token 0 sees only itself
         1  [  ✓    ✓    ✗    ✗    ✗ ]   token 1 sees 0,1
         2  [  ✓    ✓    ✓    ✗    ✗ ]   token 2 sees 0,1,2
         3  [  ✓    ✓    ✓    ✓    ✗ ]   token 3 sees 0..3
         4  [  ✓    ✓    ✓    ✓    ✓ ]   token 4 sees all
```

`✗` = True in mask = −1e9 in scores = 0 after softmax  
`✓` = False in mask = unchanged

**Shape**: `(1, 1, seq, seq)` — same for every batch item and every head.

In [ ]:
def make_causal_mask(seq_len):
    """
    Creates an upper-triangular boolean mask.

    Parameters
    ----------
    seq_len : int

    Returns
    -------
    mask : (1, 1, seq_len, seq_len)  bool
           True = this position is FORBIDDEN (future token)

    np.triu(..., k=1) gives the strict upper triangle (above the diagonal).
    The diagonal itself is allowed (a token can attend to itself).
    """
    # ones everywhere, then keep only strict upper triangle
    mask = np.triu(np.ones((seq_len, seq_len), dtype=bool), k=1)
    return mask[np.newaxis, np.newaxis, :, :]   # (1, 1, seq, seq)


# ── Demo ──────────────────────────────────────────────────────────────────────
seq_len    = 6
causal_mask = make_causal_mask(seq_len)

print("Causal mask shape:", causal_mask.shape)
print()
print("Causal mask (True = blocked future position):")
print(causal_mask[0, 0].astype(int))

# ── Visualise ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(causal_mask[0, 0].astype(float), cmap='RdYlGn_r', vmin=0, vmax=1)
ax.set_title("Causal Mask  (green=attend, red=blocked)", fontsize=11)
ax.set_xlabel("Key position (j)  — future →")
ax.set_ylabel("Query position (i)")
ax.set_xticks(range(seq_len)); ax.set_yticks(range(seq_len))

for i in range(seq_len):
    for j in range(seq_len):
        symbol = "✗" if causal_mask[0,0,i,j] else "✓"
        color  = "white" if causal_mask[0,0,i,j] else "black"
        ax.text(j, i, symbol, ha='center', va='center', color=color, fontsize=13)

plt.tight_layout()
plt.show()
print()
print("Key insight: position i can attend to positions 0 ... i (inclusive)")
print("             position i CANNOT attend to positions i+1 ... seq-1")

### 4c · Combined Mask (Decoder Self-Attention)

The decoder's self-attention layer needs **both** masks simultaneously:
- Block future tokens (causal)
- Block `<PAD>` tokens (padding)

We combine them with a logical OR:

```
combined_mask[i,j] = causal_mask[i,j]  OR  padding_mask[j]
```

A position is blocked if it's either in the future **or** it's a pad token.

In [ ]:
def make_combined_mask(token_ids, pad_id=0):
    """
    Combines causal + padding masks for decoder self-attention.

    Parameters
    ----------
    token_ids : (batch, seq)
    pad_id    : int

    Returns
    -------
    mask : (batch, 1, seq, seq)  bool
           True = forbidden (future OR pad)
    """
    batch, seq = token_ids.shape
    pad_mask    = make_padding_mask(token_ids, pad_id)    # (batch, 1, 1, seq)
    causal_mask = make_causal_mask(seq)                   # (1,     1, seq, seq)

    # Broadcasting: (batch,1,seq,seq) OR (batch,1,1,seq) → (batch,1,seq,seq)
    return causal_mask | pad_mask


# ── Demo ──────────────────────────────────────────────────────────────────────
target_ids = np.array([
    [3, 7, 2, 0, 0],   # seq=5, last 2 are <PAD>
])

combined = make_combined_mask(target_ids, pad_id=0)
print("Combined mask shape:", combined.shape)
print()
print("Combined mask (True = blocked):")
print(combined[0, 0].astype(int))

# ── Visualise all three side by side ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

pad_m     = make_padding_mask(target_ids)[0,0]      # (1, seq)
causal_m  = make_causal_mask(5)[0,0]                # (seq, seq)
combined_m= combined[0,0]                           # (seq, seq)

# Expand padding mask to 2D for display
pad_2d = np.broadcast_to(pad_m, (5, 5))

for ax, data, title in zip(axes,
    [pad_2d, causal_m, combined_m],
    ["Padding Mask", "Causal Mask", "Combined Mask (Decoder)"]):
    ax.imshow(data.astype(float), cmap='RdYlGn_r', vmin=0, vmax=1, aspect='auto')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Key pos"); ax.set_ylabel("Query pos")
    ax.set_xticks(range(5)); ax.set_yticks(range(5))
    ax.set_xticklabels(['t0','t1','t2','PAD','PAD'], fontsize=7)
    ax.set_yticklabels(['t0','t1','t2','PAD','PAD'], fontsize=7)
    for i in range(5):
        for j in range(5):
            sym   = "✗" if data[i,j] else "✓"
            color = "white" if data[i,j] else "black"
            ax.text(j, i, sym, ha='center', va='center', fontsize=10, color=color)

plt.suptitle("Three Mask Types  (✓ = attend,  ✗ = blocked)", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

---
## 5 · Multi-Head Self-Attention

### Why multiple heads?

A single attention head computes one set of Q/K/V projections and produces
one weighted average. This means it can only focus on **one type of relationship**
per forward pass.

Multiple heads let the model attend to **different aspects in parallel**:
- Head 1 might track syntactic dependencies
- Head 2 might track coreference
- Head 3 might track positional proximity

Each head operates in a lower-dimensional subspace (d_model / h),
keeping total compute constant.

### Architecture

```
Input X: (batch, seq, d_model)
        │
        ├─ Wq → Q_full: (batch, seq, d_model)
        ├─ Wk → K_full: (batch, seq, d_model)
        └─ Wv → V_full: (batch, seq, d_model)
                │
                ▼  split into h heads
        Q_h: (batch, h, seq, d_k)    d_k = d_model / h
        K_h: (batch, h, seq, d_k)
        V_h: (batch, h, seq, d_v)    d_v = d_model / h
                │
                ▼  scaled dot-product attention (in parallel across heads)
        head_i: (batch, h, seq, d_v)
                │
                ▼  concatenate all heads
        concat: (batch, seq, d_model)
                │
                ▼  Wo output projection
        output: (batch, seq, d_model)
```

### Projection matrices

| Matrix | Shape | Learnable? |
|--------|-------|-----------|
| Wq | (d_model, d_model) | ✓ |
| Wk | (d_model, d_model) | ✓ |
| Wv | (d_model, d_model) | ✓ |
| Wo | (d_model, d_model) | ✓ |

In [ ]:
class MultiHeadSelfAttention:
    """
    Multi-Head Self-Attention block — pure NumPy.

    Parameters
    ----------
    d_model : int   total embedding dimension (must be divisible by num_heads)
    num_heads : int number of attention heads

    Each head operates on dimension d_k = d_v = d_model // num_heads.
    """

    def __init__(self, d_model: int, num_heads: int):
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model   = d_model
        self.h         = num_heads
        self.d_k       = d_model // num_heads   # per-head key/query dim
        self.d_v       = d_model // num_heads   # per-head value dim

        # ── Learnable projection weights ─────────────────────────────────
        # Xavier uniform initialisation: scale = sqrt(6 / (fan_in + fan_out))
        scale = np.sqrt(6 / (d_model + d_model))
        self.Wq = np.random.uniform(-scale, scale, (d_model, d_model))
        self.Wk = np.random.uniform(-scale, scale, (d_model, d_model))
        self.Wv = np.random.uniform(-scale, scale, (d_model, d_model))
        self.Wo = np.random.uniform(-scale, scale, (d_model, d_model))

    # ── Helper: split last dim into (h, d_k) and move h forward ──────────
    def _split_heads(self, x):
        """
        (batch, seq, d_model) → (batch, h, seq, d_k)

        Reshape last dim into (h, d_k), then transpose to put h before seq.
        This lets us run attention for all heads simultaneously.
        """
        batch, seq, _ = x.shape
        x = x.reshape(batch, seq, self.h, self.d_k)  # (batch, seq, h, d_k)
        return x.transpose(0, 2, 1, 3)               # (batch, h, seq, d_k)

    # ── Helper: inverse of _split_heads ──────────────────────────────────
    def _merge_heads(self, x):
        """
        (batch, h, seq, d_v) → (batch, seq, d_model)

        Transpose h back, then flatten last two dims.
        """
        batch, h, seq, d_v = x.shape
        x = x.transpose(0, 2, 1, 3)                       # (batch, seq, h, d_v)
        return x.reshape(batch, seq, self.h * self.d_v)   # (batch, seq, d_model)

    # ── Forward pass ─────────────────────────────────────────────────────
    def forward(self, x, mask=None):
        """
        Parameters
        ----------
        x    : (batch, seq, d_model)   input sequence
        mask : (batch, 1, seq, seq) or compatible shape
               True = masked (forbidden) position

        Returns
        -------
        output       : (batch, seq, d_model)
        attn_weights : (batch, h, seq, seq)  — one weight matrix per head
        """
        batch, seq, _ = x.shape

        # ── Step 1: linear projections ────────────────────────────────────
        # x @ Wq  →  (batch, seq, d_model)
        Q = x @ self.Wq    # (batch, seq, d_model)
        K = x @ self.Wk
        V = x @ self.Wv

        # ── Step 2: split into h heads ────────────────────────────────────
        Q = self._split_heads(Q)   # (batch, h, seq, d_k)
        K = self._split_heads(K)
        V = self._split_heads(V)

        # ── Step 3: scaled dot-product attention for ALL heads at once ────
        # Shapes: Q,K → (batch,h,seq,d_k);  @ K.T → (batch,h,seq,seq)
        d_k    = Q.shape[-1]
        scores = Q @ K.transpose(0, 1, 3, 2) / np.sqrt(d_k)
        #   (batch, h, seq_q, d_k) @ (batch, h, d_k, seq_k)
        # = (batch, h, seq_q, seq_k)

        # ── Step 4: apply mask ────────────────────────────────────────────
        # mask shape (batch, 1, seq, seq) broadcasts over h dimension
        if mask is not None:
            scores = np.where(mask[:, np.newaxis, :, :] if mask.ndim == 3
                              else mask, -1e9, scores)

        # ── Step 5: softmax → attention weights ──────────────────────────
        attn_weights = softmax(scores, axis=-1)   # (batch, h, seq, seq)

        # ── Step 6: weighted sum of values ───────────────────────────────
        context = attn_weights @ V    # (batch, h, seq, d_v)

        # ── Step 7: merge heads ───────────────────────────────────────────
        context = self._merge_heads(context)   # (batch, seq, d_model)

        # ── Step 8: output projection ────────────────────────────────────
        output = context @ self.Wo    # (batch, seq, d_model)

        return output, attn_weights


mhsa = MultiHeadSelfAttention(d_model=64, num_heads=8)
print("MultiHeadSelfAttention initialised")
print(f"  d_model = {mhsa.d_model}")
print(f"  heads   = {mhsa.h}")
print(f"  d_k     = {mhsa.d_k}  (per head)")
print()
print("Weight shapes:")
shape("Wq", mhsa.Wq); shape("Wk", mhsa.Wk)
shape("Wv", mhsa.Wv); shape("Wo", mhsa.Wo)
print(f"  Total parameters: {4 * 64 * 64:,}")

## 6 · Full Shape Trace — Every Tensor Annotated

Let's run a complete forward pass with verbose shape printing so every
transformation is 100% explicit.

In [ ]:
def forward_verbose(mhsa, x, mask=None):
    """Identical logic to mhsa.forward(), but prints every intermediate shape."""
    batch, seq, d_model = x.shape
    print(f"{'─'*55}")
    print(f"INPUT")
    print(f"  x                              {x.shape}  (batch, seq, d_model)")
    print()

    # ── Projections ───────────────────────────────────────────────────────
    Q_full = x @ mhsa.Wq
    K_full = x @ mhsa.Wk
    V_full = x @ mhsa.Wv
    print("PROJECTIONS  (x @ Wq/Wk/Wv)")
    print(f"  Q_full                         {Q_full.shape}")
    print(f"  K_full                         {K_full.shape}")
    print(f"  V_full                         {V_full.shape}")
    print()

    # ── Split heads ───────────────────────────────────────────────────────
    Q = mhsa._split_heads(Q_full)
    K = mhsa._split_heads(K_full)
    V = mhsa._split_heads(V_full)
    print(f"SPLIT HEADS  (batch,seq,d_model) → (batch,h,seq,d_k)")
    print(f"  Q                              {Q.shape}  (batch, h, seq, d_k)")
    print(f"  K                              {K.shape}")
    print(f"  V                              {V.shape}")
    print()

    # ── Scores ────────────────────────────────────────────────────────────
    scores_raw   = Q @ K.transpose(0, 1, 3, 2)
    scores_scaled= scores_raw / np.sqrt(mhsa.d_k)
    print(f"ATTENTION SCORES  (Q @ Kᵀ / √d_k)")
    print(f"  scores_raw                     {scores_raw.shape}  (batch, h, seq_q, seq_k)")
    print(f"  scores_scaled                  {scores_scaled.shape}")
    print()

    # ── Mask ──────────────────────────────────────────────────────────────
    if mask is not None:
        m = mask[:, np.newaxis, :, :] if mask.ndim == 3 else mask
        print(f"MASK  shape={mask.shape}  (broadcast over h dim)")
        scores_masked = np.where(m, -1e9, scores_scaled)
        blocked = m[0,0].sum() if m.ndim==4 else 0
        print(f"  Positions masked (batch 0): {int(blocked)}")
        scores_scaled = scores_masked
        print()

    # ── Softmax ───────────────────────────────────────────────────────────
    weights = softmax(scores_scaled, axis=-1)
    print(f"ATTENTION WEIGHTS  (softmax over key dim)")
    print(f"  weights                        {weights.shape}")
    print(f"  row sum check (head 0, pos 0): {weights[0,0,0].sum():.6f}  ← must be 1.0")
    print()

    # ── Context ───────────────────────────────────────────────────────────
    context_split  = weights @ V
    context_merged = mhsa._merge_heads(context_split)
    print(f"CONTEXT VECTORS")
    print(f"  context (per head)             {context_split.shape}  (batch, h, seq, d_v)")
    print(f"  context (merged)               {context_merged.shape}  (batch, seq, d_model)")
    print()

    # ── Output projection ─────────────────────────────────────────────────
    output = context_merged @ mhsa.Wo
    print(f"OUTPUT PROJECTION  (context @ Wo)")
    print(f"  output                         {output.shape}  (batch, seq, d_model)")
    print(f"{'─'*55}")

    return output, weights


# Run the trace
batch_size, seq_len, d_model = 2, 6, 64
num_heads = 8
mhsa = MultiHeadSelfAttention(d_model=d_model, num_heads=num_heads)

x       = np.random.randn(batch_size, seq_len, d_model)
tok_ids = np.array([[4, 2, 7, 1, 0, 0],   # last 2 PAD
                    [3, 5, 8, 2, 6, 0]])   # last 1 PAD
mask    = make_padding_mask(tok_ids)       # (batch, 1, 1, seq)

output, weights = forward_verbose(mhsa, x, mask)

## 7 · Masking Correctness Checks

This is where most bugs hide. We verify each mask type mathematically.

In [ ]:
print("=" * 55)
print("MASKING CORRECTNESS CHECKS")
print("=" * 55)

# ── Check 1: PAD positions get exactly 0 attention weight ────────────────────
print("\n[CHECK 1] Padding mask — PAD tokens receive zero attention weight")
_, weights_pad = mhsa.forward(x, mask=mask)
# tok_ids[0] = [4,2,7,1,0,0] → positions 4,5 are PAD
pad_weights_received = weights_pad[0, :, :, 4:].max()  # max attention to PAD positions
print(f"  Max attention weight given to PAD positions (batch 0): {pad_weights_received:.2e}")
print(f"  {'✓ PASS' if pad_weights_received < 1e-6 else '✗ FAIL'}")

# ── Check 2: causal — upper triangle is 0 ────────────────────────────────────
print("\n[CHECK 2] Causal mask — strictly upper-triangular = zero")
causal_m  = make_causal_mask(seq_len)
_, weights_causal = mhsa.forward(x, mask=causal_m)
# Upper triangle (k=1) should be exactly 0 after softmax
upper = np.triu(weights_causal[0, 0], k=1)
print(f"  Max weight in upper triangle (head 0): {upper.max():.2e}")
print(f"  {'✓ PASS' if upper.max() < 1e-6 else '✗ FAIL'}")

# ── Check 3: lower triangle is non-zero ──────────────────────────────────────
print("\n[CHECK 3] Causal mask — lower triangle (past) has non-zero weights")
lower = np.tril(weights_causal[0, 0])
print(f"  Min weight in lower triangle: {lower[lower>0].min():.4f}")
print(f"  {'✓ PASS' if lower[lower>0].min() > 0 else '✗ FAIL'}")

# ── Check 4: all rows still sum to 1 ─────────────────────────────────────────
print("\n[CHECK 4] All attention weight rows sum to 1.0 (probability distribution)")
row_sums = weights_causal[0, 0].sum(axis=-1)
print(f"  Row sums: {np.round(row_sums, 6)}")
print(f"  {'✓ PASS' if np.allclose(row_sums, 1.0) else '✗ FAIL'}")

# ── Check 5: combined mask combines both ──────────────────────────────────────
print("\n[CHECK 5] Combined mask — PAD positions blocked AND upper triangle blocked")
combined_m = make_combined_mask(tok_ids[:1], pad_id=0)  # batch 0 only, seq=6, PAD at 4,5
_, weights_combined = mhsa.forward(x[:1], mask=combined_m)
# Positions 4,5 are PAD → should get 0 weight from all query positions
# Upper triangle should also be 0
pad_block = weights_combined[0, 0, :, 4:].max()
tri_block = np.triu(weights_combined[0, 0], k=1).max()
print(f"  Max weight to PAD positions (cols 4,5): {pad_block:.2e}")
print(f"  Max weight in upper triangle:           {tri_block:.2e}")
print(f"  {'✓ PASS' if pad_block < 1e-6 and tri_block < 1e-6 else '✗ FAIL'}")

# ── Check 6: self-attention is deterministic ──────────────────────────────────
print("\n[CHECK 6] Deterministic — same input gives same output")
out1, _ = mhsa.forward(x)
out2, _ = mhsa.forward(x)
print(f"  Max abs diff between two runs: {np.abs(out1-out2).max():.2e}")
print(f"  {'✓ PASS' if np.allclose(out1, out2) else '✗ FAIL'}")

print("\n" + "=" * 55)

## 8 · Visualising Attention Weights

Heatmaps of the attention weight matrix for each head — both with and without
the causal mask — to see exactly what each head "looks at".

In [ ]:
def plot_attention_heads(weights, title, tokens=None, max_heads=8):
    """
    weights : (batch, h, seq, seq)  — plot batch 0
    tokens  : list of token strings for axis labels
    """
    h   = min(weights.shape[1], max_heads)
    fig, axes = plt.subplots(2, h//2, figsize=(14, 6))
    axes = axes.flatten()

    for i in range(h):
        ax = axes[i]
        im = ax.imshow(weights[0, i], cmap='Blues', vmin=0, vmax=weights[0,i].max())
        ax.set_title(f'Head {i}', fontsize=9)
        if tokens:
            ax.set_xticks(range(len(tokens))); ax.set_xticklabels(tokens, rotation=45, fontsize=7)
            ax.set_yticks(range(len(tokens))); ax.set_yticklabels(tokens, fontsize=7)
        plt.colorbar(im, ax=ax, shrink=0.7)

    plt.suptitle(title, fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()


tokens = ["the", "king", "rules", "the", "realm", "now"]
x_demo = np.random.randn(1, 6, 64)

mhsa_demo = MultiHeadSelfAttention(d_model=64, num_heads=8)

# ── No mask — bidirectional (encoder style) ───────────────────────────────────
_, w_full = mhsa_demo.forward(x_demo)
plot_attention_heads(w_full, "Encoder Self-Attention (No Mask) — all heads", tokens)

# ── Causal mask — unidirectional (decoder style) ─────────────────────────────
causal_m = make_causal_mask(6)
_, w_causal = mhsa_demo.forward(x_demo, mask=causal_m)
plot_attention_heads(w_causal, "Decoder Self-Attention (Causal Mask) — all heads", tokens)

## 9 · Cross-Attention (Encoder → Decoder)

In the full Transformer, the decoder has a **cross-attention** layer between
its self-attention and feed-forward layers.

- **Q** comes from the **decoder** (what the decoder is currently generating)
- **K, V** come from the **encoder output** (the encoded source sequence)

This lets each decoder position look at the most relevant encoder positions.

```
Encoder output: (batch, src_seq, d_model)   ← fixed after encoding
Decoder state:  (batch, tgt_seq, d_model)   ← grows token by token

Cross-attention:
  Q = decoder_state @ Wq   (batch, tgt_seq, d_model)
  K = encoder_out   @ Wk   (batch, src_seq, d_model)
  V = encoder_out   @ Wv   (batch, src_seq, d_model)

  scores = Q @ Kᵀ / √dₖ   (batch, h, tgt_seq, src_seq)
  output                   (batch, tgt_seq, d_model)
```

> The **padding mask** here is built from the *encoder* sequence (to ignore
> encoder `<PAD>` tokens). There is **no causal mask** in cross-attention —
> the decoder is allowed to look at all encoder positions.

In [ ]:
class CrossAttention:
    """
    Cross-attention: Q from decoder, K and V from encoder.
    Architecturally identical to MHSA — only the inputs differ.
    """
    def __init__(self, d_model, num_heads):
        self.mhsa = MultiHeadSelfAttention(d_model, num_heads)

    def forward(self, decoder_state, encoder_output, encoder_pad_mask=None):
        """
        Parameters
        ----------
        decoder_state  : (batch, tgt_seq, d_model)
        encoder_output : (batch, src_seq, d_model)
        encoder_pad_mask : (batch, 1, 1, src_seq)  — pad mask for encoder tokens

        Returns
        -------
        output       : (batch, tgt_seq, d_model)
        attn_weights : (batch, h, tgt_seq, src_seq)
        """
        mhsa = self.mhsa
        batch, tgt_seq, _ = decoder_state.shape

        Q = decoder_state  @ mhsa.Wq   # (batch, tgt_seq, d_model)
        K = encoder_output @ mhsa.Wk   # (batch, src_seq, d_model)
        V = encoder_output @ mhsa.Wv

        Q = mhsa._split_heads(Q)   # (batch, h, tgt_seq, d_k)
        K = mhsa._split_heads(K)   # (batch, h, src_seq, d_k)
        V = mhsa._split_heads(V)

        d_k    = Q.shape[-1]
        scores = Q @ K.transpose(0, 1, 3, 2) / np.sqrt(d_k)
        # (batch, h, tgt_seq, src_seq)

        if encoder_pad_mask is not None:
            scores = np.where(encoder_pad_mask, -1e9, scores)

        weights = softmax(scores, axis=-1)
        context = mhsa._merge_heads(weights @ V)   # (batch, tgt_seq, d_model)
        output  = context @ mhsa.Wo

        return output, weights


# ── Demo ──────────────────────────────────────────────────────────────────────
src_seq, tgt_seq = 7, 5
enc_output    = np.random.randn(1, src_seq, 64)
dec_state     = np.random.randn(1, tgt_seq, 64)

src_ids       = np.array([[4, 2, 7, 1, 3, 0, 0]])   # 2 PAD tokens in encoder
enc_pad_mask  = make_padding_mask(src_ids)           # (1, 1, 1, src_seq)

cross_attn    = CrossAttention(d_model=64, num_heads=8)
ca_output, ca_weights = cross_attn.forward(dec_state, enc_output, enc_pad_mask)

print("Cross-Attention shapes:")
shape("encoder output (K, V source)", enc_output)
shape("decoder state  (Q source)",    dec_state)
shape("cross-attn weights",           ca_weights)
shape("output",                       ca_output)
print()
print(f"Weights shape: (batch={ca_weights.shape[0]}, heads={ca_weights.shape[1]},")
print(f"               tgt_seq={ca_weights.shape[2]}, src_seq={ca_weights.shape[3]})")

# Verify PAD positions get zero weight
pad_weight = ca_weights[0, :, :, 5:].max()
print(f"\nMax attention weight to encoder PAD positions: {pad_weight:.2e}")
print(f"{'✓ PASS' if pad_weight < 1e-6 else '✗ FAIL'}  — encoder PAD correctly blocked")

---
## 10 · Summary

### What we built

| Component | Key detail |
|-----------|-----------|
| **Scaled dot-product attention** | Q @ Kᵀ / √dₖ → softmax → @ V |
| **Scaling by √dₖ** | Prevents softmax saturation for large d_k |
| **Padding mask** | `(batch, 1, 1, seq_k)` — blocks `<PAD>` tokens; broadcasts over all heads and query positions |
| **Causal mask** | `(1, 1, seq, seq)` upper triangle — blocks future tokens; same for all batches/heads |
| **Combined mask** | Logical OR of both — used in decoder self-attention |
| **Head split/merge** | `(batch,seq,d_model)` ↔ `(batch,h,seq,d_k)` via reshape + transpose |
| **Output projection Wo** | Mixes information across heads after concat |
| **Cross-attention** | Q from decoder, K/V from encoder — only padding mask, no causal mask |

### Mask shape cheatsheet

```
Padding mask  :  (batch, 1, 1, seq_k)   ← broadcast over h, seq_q
Causal mask   :  (1, 1, seq, seq)        ← broadcast over batch, h
Combined mask :  (batch, 1, seq, seq)    ← explicit for all query positions
```

### What to build next
1. **Add layer normalisation** (pre-norm or post-norm)
2. **Add position-wise feed-forward network** (FFN): two linear layers + ReLU
3. **Add positional encodings** (sinusoidal or learned)
4. **Stack N encoder/decoder blocks**
5. → You now have a complete Transformer

The masking logic you built here transfers **unchanged** to any Transformer implementation.
